In [ ]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# Run this FIRST
# ============================================================

!pip install -q mlflow scikit-learn matplotlib pandas


# ============================================================
# CELL 2 — Imports
# ============================================================

import mlflow
import mlflow.sklearn

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)


# ============================================================
# CELL 3 — Load & split data
# ============================================================

data = load_breast_cancer()

X, y = data.data, data.target


# stratify=y ensures same class ratio in train & test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")


# ============================================================
# CELL 4 — Configure MLflow
# ============================================================

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("breast-cancer-classifier")


# ============================================================
# CELL 5 — Manual Logging
# ============================================================

MAX_DEPTH = 4
MIN_SAMPLES = 5
CRITERION = "gini"

with mlflow.start_run(run_name="manual-logging") as run:

    # Log parameters
    mlflow.log_param("max_depth", MAX_DEPTH)
    mlflow.log_param("min_samples_split", MIN_SAMPLES)
    mlflow.log_param("criterion", CRITERION)
    mlflow.log_param("random_state", 42)

    # Train model
    clf = DecisionTreeClassifier(
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES,
        criterion=CRITERION,
        random_state=42
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    # Log metrics
    mlflow.log_metric(
        "accuracy",
        accuracy_score(y_test, y_pred)
    )

    mlflow.log_metric(
        "precision",
        precision_score(y_test, y_pred)
    )

    mlflow.log_metric(
        "recall",
        recall_score(y_test, y_pred)
    )

    mlflow.log_metric(
        "f1",
        f1_score(y_test, y_pred)
    )

    # Save confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        ax=ax
    )

    plt.tight_layout()

    plt.savefig("confusion_matrix.png")

    mlflow.log_artifact("confusion_matrix.png")

    plt.close()

    # Save model
    mlflow.sklearn.log_model(
        clf,
        "decision-tree-model"
    )

    print(f"Run ID : {run.info.run_id}")

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")


# ============================================================
# CELL 6 — Autolog
# ============================================================

mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True
)

with mlflow.start_run(run_name="autolog-run"):

    clf2 = DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    )

    clf2.fit(X_train, y_train)

    print("autolog run complete — check the UI for logged params & metrics")


# ============================================================
# CELL 7 — Launch MLflow UI
# ============================================================

!mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000 &

print("UI running at http://localhost:5000")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.4

2026/05/29 12:11:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/29 12:11:48 INFO mlflow.store.db.utils: Updating database tables
2026/05/29 12:11:50 INFO mlflow.tracking.fluent: Experiment with name 'breast-cancer-classifier' does not exist. Creating a new experiment.
2026/05/29 12:11:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 12:11:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID : 8acd440b16db448685d0ba3669a456df
Accuracy: 0.9386


2026/05/29 12:12:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


autolog run complete — check the UI for logged params & metrics
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/29 12:12:16 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/05/29 12:12:16 INFO:     Started parent process [5348]
2026/05/29 12:12:55 INFO:     Started server process [5409]
2026/05/29 12:12:55 INFO:     Waiting for application startup.
2026/05/29 12:12:55 INFO:     Application startup complete.
2026/05/29 12:12:55 INFO mlflow.server.jobs.utils: Registered online_scoring_scheduler periodic task (runs every 1 minute)
2026/05/29 12:12:55 INFO:     Started server process [5406]
2026/05/29 12:12:55 INFO:     Waiting for application startup.
2026/05/29 12:12:55 INFO:     Application startup complete.
2026/05/29 12:12:55 INFO:     Started serv